# YOLO Fine-tuning for Road Damage Detection

Training YOLO model for road damage detection: Pothole, Alligator Crack, Transverse Crack, Longitudinal Crack

In [1]:
# Install and import required packages
%pip install ultralytics -q

import os
import yaml
import shutil
import random
import torch
from glob import glob
from ultralytics import YOLO

# Check CUDA availability
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Number of GPUs: {torch.cuda.device_count()}")

Note: you may need to restart the kernel to use updated packages.
CUDA available: True
Number of GPUs: 8
CUDA available: True
Number of GPUs: 8


In [2]:
# Load YOLO11m model
model = YOLO('yolo11m.pt')

In [3]:
# Prepare dataset for YOLO training
def prepare_yolo_dataset(data_dir="data", output_dir="yolo_dataset"):
    # Create directory structure
    for split in ['train', 'val', 'test']:
        os.makedirs(f"{output_dir}/{split}/images", exist_ok=True)
        os.makedirs(f"{output_dir}/{split}/labels", exist_ok=True)
    
    # Collect all image-label pairs from country directories
    all_pairs = []
    country_dirs = [d for d in os.listdir(data_dir) if d.startswith('country_')]
    
    for country_dir in country_dirs:
        images_path = f"{data_dir}/{country_dir}/images"
        labels_path = f"{data_dir}/{country_dir}/labels"
        
        for img_file in glob(f"{images_path}/*.jpg") + glob(f"{images_path}/*.png"):
            base_name = os.path.splitext(os.path.basename(img_file))[0]
            label_file = f"{labels_path}/{base_name}.txt"
            if os.path.exists(label_file):
                all_pairs.append((img_file, label_file))
    
    # Split dataset (70% train, 15% val, 15% test)
    random.seed(42)
    random.shuffle(all_pairs)
    
    n = len(all_pairs)
    train_end = int(0.7 * n)
    val_end = int(0.85 * n)
    
    splits = {
        'train': all_pairs[:train_end],
        'val': all_pairs[train_end:val_end],
        'test': all_pairs[val_end:]
    }
    
    # Copy files to respective directories
    for split_name, pairs in splits.items():
        for img_path, lbl_path in pairs:
            shutil.copy(img_path, f"{output_dir}/{split_name}/images/{os.path.basename(img_path)}")
            shutil.copy(lbl_path, f"{output_dir}/{split_name}/labels/{os.path.basename(lbl_path)}")
    
    # Create YAML configuration
    yaml_data = {
        'path': os.path.abspath(output_dir),
        'train': 'train/images',
        'val': 'val/images', 
        'test': 'test/images',
        'nc': 4,
        'names': ['Pothole', 'Alligator Crack', 'Transverse Crack', 'Longitudinal Crack']
    }
    
    with open(f"{output_dir}/data.yaml", 'w') as f:
        yaml.dump(yaml_data, f, sort_keys=False)
    
    return output_dir

# Prepare dataset if not exists
if not os.path.exists('yolo_dataset/data.yaml'):
    prepare_yolo_dataset()

In [ ]:
# Train YOLO model with multi-GPU support
import os

# Set up multi-GPU environment
os.environ['CUDA_VISIBLE_DEVICES'] = '1,2,3,4,5,6,7'
num_gpus = len(os.environ['CUDA_VISIBLE_DEVICES'].split(','))
base_batch_size = 16
total_batch_size = base_batch_size * num_gpus

results = model.train(
    data='yolo_dataset/data.yaml',
    epochs=100,
    imgsz=640,
    batch=total_batch_size,
    device=[1, 2, 3, 4, 5, 6, 7],
    project='runs',
    name='road_damage_detection_multigpu',
    workers=8 * num_gpus,
    patience=15,
    save_period=10,
    amp=True,
    cache=True,
    cos_lr=True,
    verbose=True
)

Using 7 GPUs: 1,2,3,4,5,6,7
Total batch size: 112 (base: 16 × 7 GPUs)
Ultralytics 8.3.169 🚀 Python-3.12.11 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 3090, 24253MiB)
                                                        CUDA:1 (NVIDIA GeForce RTX 3090, 24253MiB)
                                                        CUDA:2 (NVIDIA GeForce RTX 3090, 24253MiB)
                                                        CUDA:3 (NVIDIA GeForce RTX 3090, 24253MiB)
                                                        CUDA:4 (NVIDIA GeForce RTX 3090, 24253MiB)
                                                        CUDA:5 (NVIDIA GeForce RTX 3090, 24253MiB)
                                                        CUDA:6 (NVIDIA GeForce RTX 3090, 24253MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=112, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True

  6                  -1  1   1380352  ultralytics.nn.modules.block.C3k2            [512, 512, 1, True]           
  7                  -1  1   2360320  ultralytics.nn.modules.conv.Conv             [512, 512, 3, 2]              
  7                  -1  1   2360320  ultralytics.nn.modules.conv.Conv             [512, 512, 3, 2]              
  8                  -1  1   1380352  ultralytics.nn.modules.block.C3k2            [512, 512, 1, True]           
  8                  -1  1   1380352  ultralytics.nn.modules.block.C3k2            [512, 512, 1, True]           
  9                  -1  1    656896  ultralytics.nn.modules.block.SPPF            [512, 512, 5]                 
 10                  -1  1    990976  ultralytics.nn.modules.block.C2PSA           [512, 512, 1]                 
 11                  -1  1         0  torch.nn.modules.upsampling.Upsample         [None, 2, 'nearest']          
 12             [-1, 6]  1         0  ultralytics.nn.modules.conv.Concat           [1]  

train: Scanning /home/kwdahun/2025IMSC-Hackathon-CRACKPINK/yolo_dataset/train/labels.cache... 4227 images, 0 backgrounds, 0 corrupt: 100%|██████████| 4227/4227 [00:00<?, ?it/s]


WARNING ⚠️ cache='ram' may produce non-deterministic training results. Consider cache='disk' as a deterministic alternative if your disk space allows.


train: Caching images (4.8GB RAM): 100%|██████████| 4227/4227 [00:01<00:00, 2417.65it/s]


val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 910.3±730.7 MB/s, size: 72.3 KB)
WARNING ⚠️ cache='ram' may produce non-deterministic training results. Consider cache='disk' as a deterministic alternative if your disk space allows.


val: Scanning /home/kwdahun/2025IMSC-Hackathon-CRACKPINK/yolo_dataset/val/labels.cache... 906 images, 0 backgrounds, 0 corrupt: 100%|██████████| 906/906 [00:00<?, ?it/s]
val: Caching images (1.0GB RAM): 100%|██████████| 906/906 [00:01<00:00, 857.87it/s]


Plotting labels to runs/road_damage_detection_multigpu5/labels.jpg... 
optimizer: 'optimizer=auto' found, ignoring 'lr0=0.01' and 'momentum=0.937' and determining best 'optimizer', 'lr0' and 'momentum' automatically... 
optimizer: AdamW(lr=0.00125, momentum=0.9) with parameter groups 106 weight(decay=0.0), 113 weight(decay=0.000875), 112 bias(decay=0.0)
Image sizes 640 train, 640 val
Using 63 dataloader workers
Logging results to runs/road_damage_detection_multigpu5
Starting training for 100 epochs...

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
optimizer: 'optimizer=auto' found, ignoring 'lr0=0.01' and 'momentum=0.937' and determining best 'optimizer', 'lr0' and 'momentum' automatically... 
optimizer: AdamW(lr=0.00125, momentum=0.9) with parameter groups 106 weight(decay=0.0), 113 weight(decay=0.000875), 112 bias(decay=0.0)
Image sizes 640 train, 640 val
Using 63 dataloader workers
Logging results to runs/road_damage_detection_multigpu5
Starting train

      1/100      7.96G      2.841      5.738      2.669         65        640:  34%|███▍      | 13/38 [00:06<00:09,  2.51it/s]W0724 02:59:51.792000 960074 site-packages/torch/distributed/elastic/agent/server/api.py:719] Received 2 death signal, shutting down workers
W0724 02:59:51.794000 960074 site-packages/torch/distributed/elastic/multiprocessing/api.py:897] Sending process 960080 closing signal SIGINT
      1/100      7.96G      2.841      5.738      2.669         65        640:  34%|███▍      | 13/38 [00:06<00:12,  1.97it/s]
Process Process-18:
Process Process-8:
Process Process-7:
Traceback (most recent call last):
  File "/home/kwdahun/anaconda3/envs/yolo_env/lib/python3.12/multiprocessing/process.py", line 317, in _bootstrap
    util._exit_function()
  File "/home/kwdahun/anaconda3/envs/yolo_env/lib/python3.12/multiprocessing/util.py", line 363, in _exit_function
    _run_finalizers()
  File "/home/kwdahun/anaconda3/envs/yolo_env/lib/python3.12/multiprocessing/util.py", line 30

KeyboardInterrupt: 

In [ ]:
# Evaluate model
print("Evaluating model...")

# Load the best model from training
train_dirs = sorted(glob('runs/detect/road_damage_detection*'), key=os.path.getmtime, reverse=True)
if train_dirs:
    best_model_path = f"{train_dirs[0]}/weights/best.pt"
    print(f"Loading best model from: {best_model_path}")
    eval_model = YOLO(best_model_path)
else:
    print("Using current model for evaluation")
    eval_model = model

# Evaluate on test set
results = eval_model.val(
    data='yolo_dataset/data.yaml', 
    split='test',
    device=0,
    verbose=True
)

# Print metrics
print(f"\nmAP@0.5:      {results.box.map50:.4f}")
print(f"mAP@0.5:0.95: {results.box.map:.4f}")
print(f"Precision:    {results.box.mp:.4f}")
print(f"Recall:       {results.box.mr:.4f}")
print(f"Detailed results saved to: {results.save_dir}")

In [ ]:
# Save best model
train_dirs = sorted(glob('runs/detect/road_damage_detection*'), key=os.path.getmtime, reverse=True)
if train_dirs:
    best_model_path = f"{train_dirs[0]}/weights/best.pt"
    os.makedirs('yolo_checkpoints', exist_ok=True)
    shutil.copy(best_model_path, 'yolo_checkpoints/best_road_damage.pt')
    print(f"Model saved: yolo_checkpoints/best_road_damage.pt")

In [ ]:
# Inference on test images
from pathlib import Path
import matplotlib.pyplot as plt
import cv2
import numpy as np

# Load the best trained model
def load_best_model():
    """Load the best model from training runs"""
    train_dirs = sorted(glob('runs/detect/road_damage_detection*'), key=os.path.getmtime, reverse=True)
    if train_dirs:
        best_model_path = f"{train_dirs[0]}/weights/best.pt"
        print(f"Loading model from: {best_model_path}")
        return YOLO(best_model_path)
    elif os.path.exists('yolo_checkpoints/best_road_damage.pt'):
        print("Loading model from: yolo_checkpoints/best_road_damage.pt")
        return YOLO('yolo_checkpoints/best_road_damage.pt')
    else:
        print("No trained model found. Using base model.")
        return YOLO('yolo11m.pt')

# Example inference
if True:  # Set to True to run inference
    # Load trained model
    trained_model = load_best_model()
    
    # Example: Inference on test images
    test_images = glob('yolo_dataset/test/images/*.jpg')[:5]  # First 5 test images
    
    if test_images:
        print(f"Running inference on {len(test_images)} test images...")
        
        # Single image inference
        result = trained_model(test_images[0], device=0)
        
        # Display result
        annotated_img = result[0].plot()
        plt.figure(figsize=(12, 8))
        plt.imshow(cv2.cvtColor(annotated_img, cv2.COLOR_BGR2RGB))
        plt.title(f"Detection Result: {Path(test_images[0]).name}")
        plt.axis('off')
        plt.show()
        
        # Print detection details
        boxes = result[0].boxes
        if boxes is not None:
            class_names = ['Pothole', 'Alligator Crack', 'Transverse Crack', 'Longitudinal Crack']
            print(f"\nDetections in {Path(test_images[0]).name}:")
            for i, (cls, conf) in enumerate(zip(boxes.cls, boxes.conf)):
                print(f"  {class_names[int(cls)]}: {conf:.3f}")
        else:
            print("No detections found")
    else:
        print("No test images found. Please check the dataset.")